# 05 - Assets finales, guion del informe y defensa

Este notebook consolida las salidas del proyecto y las convierte en material util para el informe y la defensa. Su funcion es comprobar que las tablas y figuras principales existen, resumir que significa cada una y dejar una narrativa tecnica consistente.

La regla practica es simple: toda afirmacion numerica del informe debe salir de una tabla o figura generada por los notebooks. Asi se mantiene la reproducibilidad entre codigo, resultados y memoria escrita.


In [ ]:
from pathlib import Path
import json, sqlite3, re, random, math, warnings, sys, subprocess, importlib.util
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
INTERIM = DATA / 'interim'
PROCESSED = DATA / 'processed'
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
METRICS = ARTIFACTS / 'metrics'
MODELS = ARTIFACTS / 'models'
INDICES = ARTIFACTS / 'indices'
REPORTS = ARTIFACTS / 'reports'
DB = INTERIM / 'metadata.db'
SCHEMA_PATH = ROOT / 'schema.sql'
for d in [RAW, INTERIM, PROCESSED, FIGURES, METRICS, MODELS, INDICES, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Instalando {package} en este kernel...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

def dump_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('ROOT =', ROOT)


ROOT = /Users/jordiblascolozano/Documents/proyecto aprendizaje avanzado


In [ ]:
# Celda de autosuficiencia: descarga y materializa el dataset si la base local no existe.
ensure_package('datasets')
from datasets import load_dataset

TASK = 'ecthr_task_b'
DATASET_NAME = 'lex_glue'
DATASET_SUBSET = 'ecthr_b'
CACHE_DIR = RAW / 'hf_cache'

TEXT_KEYS = ('text', 'facts', 'fact', 'document', 'content')
ID_KEYS = ('case_id', 'id', 'doc_id')
YEAR_KEYS = ('year', 'decision_year', 'judgment_year', 'date', 'decision_date', 'judgment_date')

def extract_text_segments(example):
    text_value = ''
    for key in TEXT_KEYS:
        if key in example and example[key] is not None:
            text_value = example[key]
            break
    if isinstance(text_value, str):
        cleaned = text_value.strip()
        return [cleaned] if cleaned else []
    if isinstance(text_value, list):
        return [str(x).strip() for x in text_value if isinstance(x, str) and str(x).strip()]
    return []

def parse_year(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, int) and 1900 <= value <= 2100:
        return value
    if isinstance(value, float) and value.is_integer() and 1900 <= int(value) <= 2100:
        return int(value)
    if isinstance(value, str):
        m = re.search(r'(19|20)\d{2}', value)
        if m:
            return int(m.group(0))
    return None

def extract_year(example):
    for key in YEAR_KEYS:
        if key in example and example[key] is not None:
            y = parse_year(example[key])
            if y is not None:
                return y
    return None

def extract_label_ids(raw_labels, label_count=None):
    if isinstance(raw_labels, (int, float, bool)):
        v = int(raw_labels)
        return [v] if v >= 0 else []
    if not isinstance(raw_labels, (list, tuple)) or len(raw_labels) == 0:
        return []
    values = [int(v) for v in raw_labels if isinstance(v, (int, float, bool))]
    if label_count is not None and len(values) == label_count and set(values).issubset({0, 1}):
        return [i for i, v in enumerate(values) if v == 1]
    return sorted({v for v in values if v >= 0})

def build_case_id(example, task, split, row_idx):
    for key in ID_KEYS:
        if key in example and example[key] is not None:
            safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(example[key])).strip('_')
            if safe:
                return f'{task}_{split}_{safe}'
    return f'{task}_{split}_{row_idx:06d}'

def build_case_record(example, task, split, row_idx):
    case_id = build_case_id(example, task, split, row_idx)
    segments = extract_text_segments(example)
    text_full = '\n\n'.join(segments)
    n_tokens = len(re.findall(r'\w+', text_full, flags=re.UNICODE))
    return {
        'case_id': case_id,
        'task': task,
        'split': split,
        'year': extract_year(example),
        'text_full': text_full,
        'n_paragraphs': len(segments),
        'n_tokens': n_tokens,
    }, segments

def extract_label_names(dataset_dict):
    split_name = 'train' if 'train' in dataset_dict else list(dataset_dict.keys())[0]
    features = getattr(dataset_dict[split_name], 'features', None)
    if not features or 'labels' not in features:
        return []
    label_feature = features['labels']
    if hasattr(label_feature, 'feature') and hasattr(label_feature.feature, 'names'):
        return [str(x) for x in label_feature.feature.names]
    if hasattr(label_feature, 'names'):
        return [str(x) for x in label_feature.names]
    return []

def metadata_db_ready(path=DB):
    if not Path(path).exists():
        return False
    try:
        conn = sqlite3.connect(path)
        n_cases = conn.execute('SELECT COUNT(*) FROM cases').fetchone()[0]
        n_labels = conn.execute('SELECT COUNT(*) FROM case_labels').fetchone()[0]
        conn.close()
        return n_cases > 0 and n_labels > 0
    except sqlite3.Error:
        return False

def ensure_metadata_db():
    if metadata_db_ready(DB):
        print(f'Dataset ya preparado en {DB}')
        return
    print('Descargando lex_glue/ecthr_b y creando metadata.db...')
    dataset = load_dataset(DATASET_NAME, DATASET_SUBSET, cache_dir=str(CACHE_DIR))
    label_names = extract_label_names(dataset)
    label_count = len(label_names) if label_names else None
    conn = sqlite3.connect(DB)
    conn.executescript(SCHEMA_PATH.read_text(encoding='utf-8'))
    case_rows, paragraph_rows, label_rows = [], [], []
    discovered_labels = set()
    split_counts = {}
    for split_name in dataset.keys():
        split_counts[split_name] = 0
        for row_idx, example in enumerate(dataset[split_name]):
            case_record, segments = build_case_record(example, task=TASK, split=split_name, row_idx=row_idx)
            case_rows.append(case_record)
            for pidx, paragraph in enumerate(segments):
                paragraph_rows.append({'case_id': case_record['case_id'], 'paragraph_idx': pidx, 'paragraph_text': paragraph})
            labels = extract_label_ids(example.get('labels'), label_count=label_count)
            for label_id in labels:
                discovered_labels.add(label_id)
                label_rows.append({'case_id': case_record['case_id'], 'article_id': str(label_id), 'value': 1})
            split_counts[split_name] += 1
    if label_names:
        article_rows = [{'article_id': str(i), 'article_code': name, 'description': ''} for i, name in enumerate(label_names)]
    else:
        article_rows = [{'article_id': str(i), 'article_code': f'article_{i}', 'description': ''} for i in sorted(discovered_labels)]
    with conn:
        conn.executemany('INSERT OR REPLACE INTO cases VALUES (:case_id, :task, :split, :year, :text_full, :n_paragraphs, :n_tokens)', case_rows)
        conn.executemany('INSERT OR REPLACE INTO case_paragraphs VALUES (:case_id, :paragraph_idx, :paragraph_text)', paragraph_rows)
        conn.executemany('INSERT OR REPLACE INTO articles VALUES (:article_id, :article_code, :description)', article_rows)
        conn.executemany('INSERT OR REPLACE INTO case_labels VALUES (:case_id, :article_id, :value)', label_rows)
        counts = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in ['cases','case_paragraphs','articles','case_labels','experiment_runs','predictions','explanations']}
    conn.close()
    report = {
        'stage': 'auto_ingestion_from_notebook',
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'dataset': {'name': DATASET_NAME, 'subset': DATASET_SUBSET, 'cache_dir': str(CACHE_DIR)},
        'task': TASK,
        'split_counts': split_counts,
        'inserted': {'articles': len(article_rows), 'cases': len(case_rows), 'case_paragraphs': len(paragraph_rows), 'case_labels': len(label_rows)},
        'table_counts': counts,
        'sqlite_path': str(DB),
    }
    dump_json(report, REPORTS / 'ingestion_status.json')
    print('Dataset preparado:', counts)

ensure_metadata_db()


## Mapa final de evidencias

Cada bloque del proyecto responde a una pregunta distinta. Esta tabla ayuda a no mezclar objetivos durante la redaccion del informe.


In [ ]:
evidence_map = pd.DataFrame([
    {'bloque': 'EDA', 'pregunta': 'Como son los datos?', 'evidencia': 'splits, longitud, frecuencia de articulos, coocurrencia'},
    {'bloque': 'Clasificacion', 'pregunta': 'Predice articulos?', 'evidencia': 'macro-F1, micro-F1, Hamming loss'},
    {'bloque': 'Retrieval', 'pregunta': 'Recupera precedentes utiles?', 'evidencia': 'Recall@5, Recall@10, nDCG@10'},
    {'bloque': 'XAI', 'pregunta': 'Se puede auditar la decision?', 'evidencia': 'coeficientes, terminos influyentes, LIME'},
    {'bloque': 'Drift', 'pregunta': 'Cambia el problema con el tiempo?', 'evidencia': 'divergencia JS, rendimiento por split/anio'},
    {'bloque': 'Errores', 'pregunta': 'Como falla el sistema?', 'evidencia': 'falsos positivos, falsos negativos, patrones'},
])
display(evidence_map)


,bloque,pregunta,evidencia
0,EDA,Como son los datos?,"splits, longitud, frecuencia de articulos, coo..."
1,Clasificacion,Predice articulos?,"macro-F1, micro-F1, Hamming loss"
2,Retrieval,Recupera precedentes utiles?,"Recall@5, Recall@10, nDCG@10"
3,XAI,Se puede auditar la decision?,"coeficientes, terminos influyentes, LIME"
4,Drift,Cambia el problema con el tiempo?,"divergencia JS, rendimiento por split/anio"
5,Errores,Como falla el sistema?,"falsos positivos, falsos negativos, patrones"


## Tablas finales

Las tablas `paper_*.csv` son las candidatas a entrar en el informe. No todas tienen que aparecer completas: lo importante es seleccionar las que sostienen la argumentacion.

- Clasificacion: demuestra que los modelos lineales superan al baseline.
- Retrieval: demuestra que el sistema recupera precedentes con articulos compartidos.
- XAI: muestra que las predicciones se pueden inspeccionar por terminos.
- Drift: cuantifica cambios de distribucion y rendimiento.
- Errores: convierte las metricas en riesgos interpretables.


In [ ]:
final_tables = {}
for name in ['paper_classification_table.csv','paper_retrieval_table.csv','paper_xai_table.csv','paper_drift_table.csv','paper_error_pattern_table.csv','paper_error_split_table.csv']:
    path = METRICS / name
    if path.exists():
        final_tables[name] = pd.read_csv(path)
        print('\n###', name)
        display(final_tables[name].head(20))
    else:
        print('Falta', name)



### paper_classification_table.csv


,model,split,n_cases,macro_f1,micro_f1,hamming_loss
0,tfidf_logreg_threshold_tuned,test,1000,0.672070,0.736325,0.075200
1,tfidf_logreg_threshold_tuned,train,9000,0.816111,0.876243,0.037333
2,tfidf_logreg_threshold_tuned,validation,1000,0.654485,0.758156,0.068200



### paper_retrieval_table.csv


,retrieval_method,split,n_queries,n_queries_with_relevant,ndcg_at_10,recall_at_5,recall_at_10
0,tfidf_cosine_sampled,validation,250,241,0.883109,0.948,0.964
1,tfidf_cosine_sampled,test,250,242,0.867210,0.936,0.968
2,tfidf_svd_cosine_sampled,validation,250,240,0.891579,0.956,0.960
3,tfidf_svd_cosine_sampled,test,250,239,0.870227,0.928,0.956



### paper_xai_table.csv


...

,target_split,js_divergence,l1_distance,cosine_similarity,delta_macro_f1,delta_micro_f1,delta_hamming_loss
0,validation,0.017336,0.278045,0.958946,0.000000,0.000000,0.000
1,test,0.025531,0.320362,0.950533,0.017585,-0.021831,0.007



### paper_error_pattern_table.csv


,split,fp,fn,n_cases
0,test,0,0,462
1,test,1,0,194
2,test,0,1,167
3,test,1,1,85
4,test,0,2,38
5,test,2,0,26
6,test,1,2,8
7,test,2,1,6
8,test,0,3,4
9,test,1,3,3



### paper_error_split_table.csv


,split,tp,fp,fn,n_errors
0,test,1.050000,0.367000,0.385000,0.752000
1,train,1.321667,0.232222,0.141111,0.373333
2,validation,1.069000,0.360000,0.322000,0.682000


## Figuras disponibles

Las figuras de `artifacts/figures/` se generan desde los notebooks, no a mano. Para el informe se recomiendan las que apoyan una idea concreta: frecuencia de etiquetas para desbalanceo, macro-F1 para comparacion de modelos, Recall@10 para retrieval, divergencia JS para drift y patrones de error para discusion critica.


In [ ]:
for p in sorted(FIGURES.glob('paper_*.png')):
    print(p.name)


paper_classification_macro_f1.png
paper_drift_js_divergence.png
paper_error_patterns.png
paper_retrieval_recall10.png


## Estructura recomendada del informe

1. **Introduccion**: problema real, usuarios, riesgos y alcance. El sistema apoya el issue spotting y la busqueda de precedentes.
2. **Trabajo relacionado**: legal NLP, representaciones TF-IDF, clasificadores lineales, XAI y drift.
3. **Datos y problema**: ECtHR Task B como clasificacion multietiqueta de articulos.
4. **Metodologia**: normalizacion, matriz `Y`, TF-IDF, One-vs-Rest, umbrales, retrieval, XAI y drift.
5. **Resultados**: clasificacion, retrieval, explicabilidad, deriva y errores.
6. **Discusion**: limitaciones juridicas, falsos negativos, falsos positivos, anclaje por precedentes y supervision humana.
7. **Conclusiones**: sistema util como apoyo, no como automatizacion del razonamiento juridico.

La seccion de metodologia debe explicar el por que de cada decision. TF-IDF se usa porque representa documentos completos sin truncar; One-vs-Rest se usa porque la salida es multietiqueta; los umbrales por etiqueta se usan porque los articulos tienen frecuencias distintas; macro-F1 se usa porque el desbalanceo hace insuficiente mirar solo metricas globales.


## Guion de defensa oral

- Empezar con el problema real: un abogado necesita priorizar articulos y precedentes.
- Aclarar el alcance: el sistema no decide por el juez ni sustituye al abogado.
- Explicar el dataset: ECtHR Task B, texto juridico largo y salida multietiqueta.
- Explicar la representacion: cada `text_full` se transforma en una fila TF-IDF con palabras y bigramas.
- Explicar One-vs-Rest: un clasificador binario por articulo, todos sobre la misma matriz `X`.
- Explicar umbrales: cada articulo tiene su propio umbral ajustado en validation.
- Mostrar resultados: baseline frente a regresion logistica, SVM y SVM con umbrales.
- Presentar retrieval como capa de utilidad real: precedentes similares evaluados con Recall@5, Recall@10 y nDCG@10.
- Presentar XAI y drift como problematicas principales de la asignatura.
- Cerrar con limitaciones: las palabras clave no son razonamiento juridico, retrieval puede anclar la busqueda y siempre hace falta supervision humana.

Frase sintetica para defensa:

> Nuestro modelo no lee el caso como una red neuronal secuencial, sino como una representacion vectorial TF-IDF. Cada caso se convierte en una fila de una matriz dispersa donde las columnas son terminos y bigramas juridicamente informativos. Sobre esa matriz entrenamos clasificadores lineales One-vs-Rest, uno por articulo. La salida final es multietiqueta, porque un caso puede activar varios articulos simultaneamente. Para mejorar la sensibilidad por articulo, ajustamos umbrales individuales usando el conjunto de validacion.


In [ ]:
summary = {
    'stage': 'final_assets_notebook',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'tables': sorted([p.name for p in METRICS.glob('paper_*.csv')]),
    'figures': sorted([p.name for p in FIGURES.glob('paper_*.png')]),
    'message': 'Proyecto convertido a flujo reproducible basado en notebooks.'
}
dump_json(summary, REPORTS / 'final_notebook_assets_status.json')
summary


{'stage': 'final_assets_notebook',
 'timestamp_utc': '2026-04-28T11:51:47.745339+00:00',
 'tables': ['paper_classification_table.csv',
  'paper_drift_table.csv',
  'paper_error_pattern_table.csv',
  'paper_error_split_table.csv',
  'paper_retrieval_table.csv',
  'paper_xai_table.csv'],
 'figures': ['paper_classification_macro_f1.png',
  'paper_drift_js_divergence.png',
  'paper_error_patterns.png',
  'paper_retrieval_recall10.png'],
 'message': 'Proyecto convertido a flujo reproducible basado en notebooks.'}